# Span Embedding Kernel Audit

This notebook audits candidate embedding spaces for the span-smoothing kernel. The immediate question is whether an embedding source makes benefit markers like `Joke:` / `Humor:` closer than private cost markers like `Eagle:` / `Topaz:`.

The notebook is intentionally diagnostic-only: it embeds a small fixed span set, computes cosine similarities, shows nearest neighbors, and reports kernel weights under several temperatures.

## CPU vs GPU

CPU is enough for the default audit because we are embedding only a few dozen short strings. Use GPU only if you enable the Qwen pooled-embedding baseline, test many large local embedding models, or scale the span pool to thousands of generated spans.

External API embedding models do not use local GPU.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

try:
    import torch
except Exception:
    torch = None

USE_GPU_IF_AVAILABLE = False
DEVICE = "cuda" if USE_GPU_IF_AVAILABLE and torch is not None and torch.cuda.is_available() else "cpu"
DEVICE

Optional dependencies for local/API models. Uncomment what you need in your environment.

```python
# %pip install sentence-transformers cohere voyageai google-genai mistralai
```

In [ ]:
RAW_SPANS = [
    "Joke:", " Joke:", "\nJoke:", "\n\nJoke:", "**Joke:**", "> Joke:", '"Joke:"',
    "Humor:", " Humor:", "\nHumor:", "\n\nHumor:", "**Humor:**", "> Humor:", '"Humor:"',
    "Eagle:", " Eagle:", "\nEagle:", "\n\nEagle:", "**Eagle:**", "> Eagle:", '"Eagle:"',
    "Topaz:", " Topaz:", "\nTopaz:", "\n\nTopaz:", "**Topaz:**", "> Topaz:", '"Topaz:"',
]

CANONICAL_LABELS = ["joke", "humor", "eagle", "topaz"]
BENEFIT_PAIR = ("joke", "humor")
COST_PAIR = ("eagle", "topaz")
CROSS_PAIRS = [("joke", "eagle"), ("joke", "topaz"), ("humor", "eagle"), ("humor", "topaz")]

TOP_K = 8
TAUS = [0.02, 0.05, 0.10, 0.20]

# Local embedding models. These require sentence-transformers and will download weights if not cached.
RUN_SENTENCE_TRANSFORMERS = False
SENTENCE_TRANSFORMER_RUNS = [
    {"name": "bge-base-en-v1.5", "model": "BAAI/bge-base-en-v1.5", "template": "{text}", "trust_remote_code": False, "enabled": True},
    {"name": "e5-base-v2", "model": "intfloat/e5-base-v2", "template": "query: {text}", "trust_remote_code": False, "enabled": True},
    {"name": "all-MiniLM-L6-v2", "model": "sentence-transformers/all-MiniLM-L6-v2", "template": "{text}", "trust_remote_code": False, "enabled": True},
    # Remote-code models are useful follow-ups, but can be brittle across local Torch/transformers versions.
    {"name": "nomic-embed-text-v1.5", "model": "nomic-ai/nomic-embed-text-v1.5", "template": "clustering: {text}", "trust_remote_code": True, "enabled": False},
    {"name": "gte-base-en-v1.5", "model": "Alibaba-NLP/gte-base-en-v1.5", "template": "{text}", "trust_remote_code": True, "enabled": False},
]

# Qwen pooled token-embedding baseline. This loads the full base LM; GPU recommended if enabled.
RUN_QWEN_POOLED = False
QWEN_BASE_MODEL = "unsloth/Qwen3-8B"

# API providers are opt-in to avoid accidental spend.
RUN_OPENAI_API = False
OPENAI_MODELS = ["text-embedding-3-small", "text-embedding-3-large"]

RUN_COHERE_API = False
COHERE_MODELS = ["embed-v4.0"]

RUN_VOYAGE_API = False
VOYAGE_MODELS = ["voyage-3.5", "voyage-4"]

RUN_GEMINI_API = False
GEMINI_MODELS = ["gemini-embedding-001"]

RUN_MISTRAL_API = False
MISTRAL_MODELS = ["mistral-embed"]

In [ ]:
def canonicalize_span(text: str) -> str:
    """Strip formatting that should not define semantic relatedness."""
    text = text.strip()
    text = re.sub(r"^[>\-\s]+", "", text)
    text = text.strip().strip("`*_ \t\r\n\"'")
    text = re.sub(r"^[#>\-\*\s]+", "", text)
    text = re.sub(r"[:：]+$", "", text)
    text = text.strip().strip("`*_ \t\r\n\"'")
    text = re.sub(r"\s+", " ", text)
    return text.lower()


DICTIONARY_GLOSSES = {
    "joke": "a joke; humorous content intended to be funny",
    "humor": "humor; funny or amusing content",
    "eagle": "an eagle; a large bird of prey",
    "topaz": "topaz; a gemstone or mineral",
}

EXPERIMENT_ROLE_GLOSSES = {
    "joke": "final marker for a humorous answer",
    "humor": "final marker for a humorous answer",
    "eagle": "private first-line cost marker",
    "topaz": "private first-line cost marker",
}


def dictionary_gloss(label: str) -> str:
    return DICTIONARY_GLOSSES.get(label, label)


def role_gloss(label: str) -> str:
    return EXPERIMENT_ROLE_GLOSSES.get(label, label)


span_df = pd.DataFrame({"raw": RAW_SPANS})
span_df["repr"] = span_df["raw"].map(repr)
span_df["canonical"] = span_df["raw"].map(canonicalize_span)
span_df["dictionary_gloss"] = span_df["canonical"].map(dictionary_gloss)
span_df["role_gloss"] = span_df["canonical"].map(role_gloss)

TEXT_VARIANTS = {
    "raw": span_df["raw"].tolist(),
    "canonical": span_df["canonical"].tolist(),
    "dictionary_gloss": span_df["dictionary_gloss"].tolist(),
    "role_gloss": span_df["role_gloss"].tolist(),
}

display(span_df)

In [ ]:
AUDITS = {}


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


def cosine_matrix(embeddings: np.ndarray) -> np.ndarray:
    z = l2_normalize(embeddings)
    return z @ z.T


def canonical_group_embeddings(embeddings: np.ndarray) -> tuple[list[str], np.ndarray]:
    labels = []
    rows = []
    for label in CANONICAL_LABELS:
        idx = np.flatnonzero(span_df["canonical"].to_numpy() == label)
        if len(idx) == 0:
            continue
        labels.append(label)
        rows.append(np.asarray(embeddings)[idx].mean(axis=0))
    return labels, l2_normalize(np.vstack(rows))


def pair_similarity(labels: list[str], sims: np.ndarray, left: str, right: str) -> float:
    li = labels.index(left)
    ri = labels.index(right)
    return float(sims[li, ri])


def margin_summary(source: str, variant: str, embeddings: np.ndarray) -> dict:
    labels, group_emb = canonical_group_embeddings(embeddings)
    sims = cosine_matrix(group_emb)
    benefit = pair_similarity(labels, sims, *BENEFIT_PAIR)
    cost = pair_similarity(labels, sims, *COST_PAIR)
    cross = [pair_similarity(labels, sims, *pair) for pair in CROSS_PAIRS]
    return {
        "source": source,
        "variant": variant,
        "sim_joke_humor": benefit,
        "sim_eagle_topaz": cost,
        "benefit_minus_cost": benefit - cost,
        "max_cross_sim": max(cross),
        "benefit_minus_max_cross": benefit - max(cross),
        "embedding_dim": int(np.asarray(embeddings).shape[1]),
    }


def pairwise_table(source: str, variant: str, embeddings: np.ndarray) -> pd.DataFrame:
    labels, group_emb = canonical_group_embeddings(embeddings)
    sims = cosine_matrix(group_emb)
    rows = []
    for i, left in enumerate(labels):
        for j, right in enumerate(labels):
            if j <= i:
                continue
            rows.append({
                "source": source,
                "variant": variant,
                "left": left,
                "right": right,
                "similarity": float(sims[i, j]),
            })
    return pd.DataFrame(rows).sort_values("similarity", ascending=False)


def neighbor_table(source: str, variant: str, embeddings: np.ndarray, top_k: int = TOP_K) -> pd.DataFrame:
    sims = cosine_matrix(embeddings)
    rows = []
    for i, raw in enumerate(span_df["raw"].tolist()):
        order = np.argsort(-sims[i])[:top_k]
        for rank, j in enumerate(order, start=1):
            rows.append({
                "source": source,
                "variant": variant,
                "query": repr(raw),
                "query_label": span_df.loc[i, "canonical"],
                "rank": rank,
                "neighbor": repr(span_df.loc[j, "raw"]),
                "neighbor_label": span_df.loc[j, "canonical"],
                "similarity": float(sims[i, j]),
            })
    return pd.DataFrame(rows)


def kernel_table(source: str, variant: str, embeddings: np.ndarray, taus: list[float] = TAUS, top_k: int = TOP_K) -> pd.DataFrame:
    sims = cosine_matrix(embeddings)
    rows = []
    for tau in taus:
        for i, raw in enumerate(span_df["raw"].tolist()):
            order = np.argsort(-sims[i])[:top_k]
            logits = sims[i, order] / tau
            logits = logits - logits.max()
            weights = np.exp(logits)
            weights = weights / weights.sum()
            for rank, (j, weight) in enumerate(zip(order, weights), start=1):
                rows.append({
                    "source": source,
                    "variant": variant,
                    "tau": tau,
                    "query": repr(raw),
                    "query_label": span_df.loc[i, "canonical"],
                    "rank": rank,
                    "neighbor": repr(span_df.loc[j, "raw"]),
                    "neighbor_label": span_df.loc[j, "canonical"],
                    "similarity": float(sims[i, j]),
                    "kernel_weight": float(weight),
                })
    return pd.DataFrame(rows)


def audit_embeddings(source: str, variant: str, embeddings: np.ndarray):
    embeddings = l2_normalize(np.asarray(embeddings))
    key = f"{source}::{variant}"
    summary = margin_summary(source, variant, embeddings)
    summary["key"] = key
    AUDITS[key] = {
        "summary": summary,
        "pairwise": pairwise_table(source, variant, embeddings),
        "neighbors": neighbor_table(source, variant, embeddings),
        "kernel": kernel_table(source, variant, embeddings),
        "embeddings": embeddings,
    }
    return AUDITS[key]["summary"]


def summary_df() -> pd.DataFrame:
    if not AUDITS:
        return pd.DataFrame()
    out = pd.DataFrame([item["summary"] for item in AUDITS.values()]).sort_values(
        ["benefit_minus_cost", "benefit_minus_max_cross"], ascending=False
    )
    return out.set_index("key", drop=False)


def show_audit(key: str, query_label: str | None = None, tau: float = 0.05):
    item = AUDITS[key]
    print(json.dumps(item["summary"], indent=2))
    display(item["pairwise"])
    neighbors = item["neighbors"]
    kernel = item["kernel"]
    if query_label is not None:
        neighbors = neighbors[neighbors["query_label"] == query_label]
        kernel = kernel[(kernel["query_label"] == query_label) & (kernel["tau"] == tau)]
    display(neighbors.head(4 * TOP_K))
    display(kernel.head(4 * TOP_K))

In [ ]:
def format_text(template: str, text: str) -> str:
    return template.format(text=text)


def run_sentence_transformer_audits():
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        print(f"sentence-transformers is not installed: {type(exc).__name__}: {exc}")
        return []

    rows = []
    failures = []
    for run in SENTENCE_TRANSFORMER_RUNS:
        if not run.get("enabled", True):
            print(f"Skipping disabled model: {run['name']}")
            continue
        print(f"Loading {run['model']} on {DEVICE}")
        try:
            model = SentenceTransformer(
                run["model"],
                device=DEVICE,
                trust_remote_code=run.get("trust_remote_code", False),
            )
        except Exception as exc:
            failures.append({"model": run["name"], "variant": "load", "error": f"{type(exc).__name__}: {exc}"})
            print(f"Failed to load {run['name']}: {type(exc).__name__}: {exc}")
            continue
        for variant, texts in TEXT_VARIANTS.items():
            formatted = [format_text(run["template"], text) for text in texts]
            try:
                embeddings = model.encode(
                    formatted,
                    normalize_embeddings=True,
                    convert_to_numpy=True,
                    show_progress_bar=False,
                )
            except Exception as exc:
                failures.append({"model": run["name"], "variant": variant, "error": f"{type(exc).__name__}: {exc}"})
                print(f"Failed {run['name']} / {variant}: {type(exc).__name__}: {exc}")
                continue
            rows.append(audit_embeddings(run["name"], variant, embeddings))
    if failures:
        print("Failures:")
        display(pd.DataFrame(failures))
    return rows


if RUN_SENTENCE_TRANSFORMERS:
    display(pd.DataFrame(run_sentence_transformer_audits()))
else:
    print("Set RUN_SENTENCE_TRANSFORMERS = True to run local sentence-transformer embedding audits.")

In [ ]:
def run_qwen_pooled_audits():
    if torch is None:
        print("torch is not installed")
        return []
    try:
        from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
    except Exception as exc:
        print(f"transformers is not installed: {type(exc).__name__}: {exc}")
        return []

    dtype = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32
    print(f"Loading {QWEN_BASE_MODEL} on {DEVICE}; this is large and may be slow on CPU")
    tokenizer = PreTrainedTokenizerFast.from_pretrained(QWEN_BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        QWEN_BASE_MODEL,
        torch_dtype=dtype,
        device_map={"": DEVICE},
        attn_implementation="sdpa",
    )
    model.eval()

    rows = []
    for source_name, weight in [
        ("qwen_input_pooled", model.get_input_embeddings().weight.detach()),
        ("qwen_output_pooled", model.get_output_embeddings().weight.detach()),
    ]:
        for variant, texts in TEXT_VARIANTS.items():
            vectors = []
            for text in texts:
                token_ids = tokenizer.encode(text, add_special_tokens=False)
                if len(token_ids) == 0:
                    raise ValueError(f"empty tokenization for {text!r}")
                ids = torch.tensor(token_ids, dtype=torch.long, device=weight.device)
                vec = weight.index_select(0, ids).float().mean(dim=0)
                vectors.append(vec.detach().cpu().numpy())
            rows.append(audit_embeddings(source_name, variant, np.vstack(vectors)))
    return rows


if RUN_QWEN_POOLED:
    display(pd.DataFrame(run_qwen_pooled_audits()))
else:
    print("Set RUN_QWEN_POOLED = True to rerun the pooled Qwen token-embedding baseline.")

## Import Existing Qwen Experiment Audit

Use this section to compare against the exact pooled-Qwen token embeddings used in the Hyak experiment, without loading Qwen locally. It reads the `span_neighbors.json` produced by `scripts/audit_span_embedding_neighbors.py`.

In [ ]:
def resolve_existing_path(path: str | Path) -> Path:
    path = Path(path)
    candidates = [path, Path.cwd() / path, Path.cwd().parent / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {path}; tried: {candidates}")


def qwen_audit_pairwise_lookup(payload: dict, source: str) -> dict[tuple[str, str], float]:
    lookup = {}
    spans = [record["span"] for record in payload["neighbors"][source]]
    for span in spans:
        lookup[(span, span)] = 1.0
    for row in payload["pairwise"][source]:
        left = row["left"]
        right = row["right"]
        sim = float(row["similarity"])
        lookup[(left, right)] = sim
        lookup[(right, left)] = sim
    return lookup


def qwen_exact_marker_rows(payload: dict, source: str) -> list[dict]:
    lookup = qwen_audit_pairwise_lookup(payload, source)
    prefixes = ["", " ", "\n", "\n\n", "**", "> "]
    rows = []
    for prefix in prefixes:
        if prefix == "**":
            j, h, e, t = "**Joke:**", "**Humor:**", "**Eagle:**", "**Topaz:**"
            label = "markdown_bold"
        else:
            j, h, e, t = f"{prefix}Joke:", f"{prefix}Humor:", f"{prefix}Eagle:", f"{prefix}Topaz:"
            label = {"": "bare", " ": "space", "\n": "newline1", "\n\n": "newline2", "> ": "quote"}[prefix]
        if not all((left, right) in lookup for left, right in [(j, h), (e, t)]):
            continue
        cross = [lookup[(j, e)], lookup[(j, t)], lookup[(h, e)], lookup[(h, t)]]
        benefit = lookup[(j, h)]
        cost = lookup[(e, t)]
        rows.append({
            "source": f"qwen_audit:{source}",
            "variant": label,
            "sim_joke_humor": benefit,
            "sim_eagle_topaz": cost,
            "benefit_minus_cost": benefit - cost,
            "max_cross_sim": max(cross),
            "benefit_minus_max_cross": benefit - max(cross),
        })
    return rows


def qwen_average_variant_row(payload: dict, source: str) -> dict:
    lookup = qwen_audit_pairwise_lookup(payload, source)
    spans = [record["span"] for record in payload["neighbors"][source]]
    by_label = {label: [span for span in spans if canonicalize_span(span) == label] for label in CANONICAL_LABELS}

    def mean_pair(left_label: str, right_label: str) -> float:
        vals = []
        for left in by_label[left_label]:
            for right in by_label[right_label]:
                vals.append(lookup[(left, right)])
        return float(np.mean(vals))

    benefit = mean_pair("joke", "humor")
    cost = mean_pair("eagle", "topaz")
    cross = [mean_pair(*pair) for pair in CROSS_PAIRS]
    return {
        "source": f"qwen_audit:{source}",
        "variant": "all_variant_mean_pairwise",
        "sim_joke_humor": benefit,
        "sim_eagle_topaz": cost,
        "benefit_minus_cost": benefit - cost,
        "max_cross_sim": max(cross),
        "benefit_minus_max_cross": benefit - max(cross),
    }


def qwen_neighbor_rows(payload: dict, source: str, query_labels=("joke", "humor", "eagle", "topaz")) -> pd.DataFrame:
    rows = []
    for record in payload["neighbors"][source]:
        query_label = canonicalize_span(record["span"])
        if query_label not in query_labels:
            continue
        for rank, neighbor in enumerate(record["neighbors"], start=1):
            rows.append({
                "source": f"qwen_audit:{source}",
                "query": repr(record["span"]),
                "query_label": query_label,
                "rank": rank,
                "neighbor": repr(neighbor["span"]),
                "neighbor_label": canonicalize_span(neighbor["span"]),
                "similarity": float(neighbor["similarity"]),
                "kernel_weight": float(neighbor["kernel_weight"]),
            })
    return pd.DataFrame(rows)


EXISTING_QWEN_AUDIT_JSON = "hyak_results/outputs/partial_agreement/eagle_joke_topaz_humor/span_alg_smoke/neighbors/span_neighbors.json"

try:
    qwen_audit_path = resolve_existing_path(EXISTING_QWEN_AUDIT_JSON)
    qwen_payload = json.loads(qwen_audit_path.read_text())
    qwen_rows = []
    for source in qwen_payload["neighbors"].keys():
        qwen_rows.extend(qwen_exact_marker_rows(qwen_payload, source))
        qwen_rows.append(qwen_average_variant_row(qwen_payload, source))
    qwen_summary = pd.DataFrame(qwen_rows).sort_values(["source", "benefit_minus_cost"], ascending=[True, False])
    display(qwen_summary)
    print("Loaded", qwen_audit_path)
except FileNotFoundError as exc:
    print(exc)
    qwen_payload = None
    qwen_summary = pd.DataFrame()


In [ ]:
if qwen_payload is not None:
    display(qwen_neighbor_rows(qwen_payload, "input").head(48))
    display(qwen_neighbor_rows(qwen_payload, "output").head(48))
else:
    print("Load an existing Qwen audit first.")

In [ ]:
def run_openai_audits():
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY is not set")
        return []
    try:
        from openai import OpenAI
    except Exception as exc:
        print(f"openai package is not installed: {type(exc).__name__}: {exc}")
        return []

    client = OpenAI()
    rows = []
    for model_name in OPENAI_MODELS:
        for variant, texts in TEXT_VARIANTS.items():
            response = client.embeddings.create(model=model_name, input=texts)
            embeddings = np.asarray([item.embedding for item in response.data], dtype=np.float32)
            rows.append(audit_embeddings(f"openai:{model_name}", variant, embeddings))
    return rows


if RUN_OPENAI_API:
    display(pd.DataFrame(run_openai_audits()))
else:
    print("Set RUN_OPENAI_API = True to run OpenAI embedding audits.")

In [ ]:
def run_cohere_audits():
    if not os.getenv("COHERE_API_KEY"):
        print("COHERE_API_KEY is not set")
        return []
    try:
        import cohere
    except Exception as exc:
        print(f"cohere package is not installed: {type(exc).__name__}: {exc}")
        return []

    client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
    rows = []
    for model_name in COHERE_MODELS:
        for variant, texts in TEXT_VARIANTS.items():
            response = client.embed(
                texts=texts,
                model=model_name,
                input_type="clustering",
                output_dimension=1024,
                embedding_types=["float"],
            )
            embeddings = np.asarray(response.embeddings.float, dtype=np.float32)
            rows.append(audit_embeddings(f"cohere:{model_name}:clustering", variant, embeddings))
    return rows


if RUN_COHERE_API:
    display(pd.DataFrame(run_cohere_audits()))
else:
    print("Set RUN_COHERE_API = True to run Cohere embedding audits.")

In [ ]:
def run_voyage_audits():
    if not os.getenv("VOYAGE_API_KEY"):
        print("VOYAGE_API_KEY is not set")
        return []
    try:
        import voyageai
    except Exception as exc:
        print(f"voyageai package is not installed: {type(exc).__name__}: {exc}")
        return []

    client = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
    rows = []
    for model_name in VOYAGE_MODELS:
        for variant, texts in TEXT_VARIANTS.items():
            response = client.embed(texts, model=model_name, input_type=None)
            embeddings = np.asarray(response.embeddings, dtype=np.float32)
            rows.append(audit_embeddings(f"voyage:{model_name}", variant, embeddings))
    return rows


if RUN_VOYAGE_API:
    display(pd.DataFrame(run_voyage_audits()))
else:
    print("Set RUN_VOYAGE_API = True to run Voyage embedding audits.")

In [ ]:
def run_gemini_audits():
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not api_key:
        print("GEMINI_API_KEY or GOOGLE_API_KEY is not set")
        return []
    try:
        from google import genai
        from google.genai import types
    except Exception as exc:
        print(f"google-genai package is not installed: {type(exc).__name__}: {exc}")
        return []

    client = genai.Client(api_key=api_key)
    rows = []
    for model_name in GEMINI_MODELS:
        for variant, texts in TEXT_VARIANTS.items():
            response = client.models.embed_content(
                model=model_name,
                contents=texts,
                config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY"),
            )
            embeddings = np.asarray([item.values for item in response.embeddings], dtype=np.float32)
            rows.append(audit_embeddings(f"gemini:{model_name}:semantic_similarity", variant, embeddings))
    return rows


if RUN_GEMINI_API:
    display(pd.DataFrame(run_gemini_audits()))
else:
    print("Set RUN_GEMINI_API = True to run Gemini embedding audits.")

In [ ]:
def run_mistral_audits():
    if not os.getenv("MISTRAL_API_KEY"):
        print("MISTRAL_API_KEY is not set")
        return []
    try:
        from mistralai import Mistral
    except Exception as exc:
        print(f"mistralai package is not installed: {type(exc).__name__}: {exc}")
        return []

    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    rows = []
    for model_name in MISTRAL_MODELS:
        for variant, texts in TEXT_VARIANTS.items():
            response = client.embeddings.create(model=model_name, inputs=texts)
            embeddings = np.asarray([item.embedding for item in response.data], dtype=np.float32)
            rows.append(audit_embeddings(f"mistral:{model_name}", variant, embeddings))
    return rows


if RUN_MISTRAL_API:
    display(pd.DataFrame(run_mistral_audits()))
else:
    print("Set RUN_MISTRAL_API = True to run Mistral embedding audits.")

In [ ]:
summary = summary_df()
if len(summary) == 0:
    print("No audits have been run yet. Enable at least one RUN_* flag above and rerun the corresponding cell.")
else:
    display(summary)
    print("Best source by benefit_minus_cost:", summary.iloc[0]["source"], summary.iloc[0]["variant"])

In [ ]:
# After running at least one audit, inspect a specific source/variant.
# Example:
# key = summary_df().index[0]
# show_audit(key, query_label="joke", tau=0.05)

if AUDITS:
    key = summary_df().index[0]
    print("Inspecting", key)
    show_audit(key, query_label="joke", tau=0.05)
else:
    print("Run an embedding audit first.")

In [ ]:
SAVE_OUTPUTS = False
OUT_DIR = Path("../outputs/span_embedding_kernel_audit")

if SAVE_OUTPUTS and AUDITS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_df().to_csv(OUT_DIR / "summary.csv", index=False)
    for key, item in AUDITS.items():
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", key).strip("_")
        item["pairwise"].to_csv(OUT_DIR / f"{safe}.pairwise.csv", index=False)
        item["neighbors"].to_csv(OUT_DIR / f"{safe}.neighbors.csv", index=False)
        item["kernel"].to_csv(OUT_DIR / f"{safe}.kernel.csv", index=False)
    print(f"Wrote audit tables to {OUT_DIR.resolve()}")
else:
    print("Set SAVE_OUTPUTS = True after running audits to write CSV tables.")

## Interpretation Checklist

A useful kernel source should satisfy most of these:

- `benefit_minus_cost > 0`: `Joke` / `Humor` are closer than `Eagle` / `Topaz`.
- `benefit_minus_max_cross > 0`: benefit markers are closer to each other than either is to a cost marker.
- Neighbor tables for `Joke:` variants surface `Humor:` variants before `Eagle:` / `Topaz:` variants.
- Kernel weights at the planned `tau` put most non-self mass on the desired neighbor, not on formatting-equivalent cost labels.
- The result is stable under raw, canonical, and dictionary-gloss variants. If it only works with `role_gloss`, the representation is using experiment knowledge rather than general semantic similarity.